# Data Owner 4 (DO4) - LoRA Training Client

This notebook is pre-configured for **DO 4 (LoRA)** participating in federated SAM2LoRA training.

## Client Configuration
- **Type**: LoRA training (full gradient-based fine-tuning)
- **Training samples**: 14
- **Test samples**: 4
- **LoRA Rank**: 16 (enhanced capacity)
- **Learning Rate**: 1e-4
- **Contribution**: YES - contributes weights to FedAvg aggregation

## Enhanced Training Features
- Data augmentation (flip, brightness, contrast, gamma)
- FedProx regularization for stability
- Background point prompts for better segmentation
- Early stopping (patience=3)

## Prerequisites
1. Go to https://colab.research.google.com/
2. Upload this notebook with `File` -> `Upload Notebook`

## Install Dependencies

In [ ]:
!uv pip install -v "git+https://github.com/OpenMined/syft-flwr.git@feat/syft-client-p2p" 2>&1 | grep -E "(OpenMined/syft-flwr|OpenMined/syft-client).*[0-9a-f]{7}"

## Login to Datasite

In [ ]:
import syft_client as sc
import syft_flwr

print(f"{sc.__version__ = }")
print(f"{syft_flwr.__version__ = }")

# do_email = input("Enter DO4's email: ")
do_email = "your.do4.lora@gmail.com"  # Replace with your email
do_client = sc.login_do(email=do_email)

## View Peers

In [ ]:
do_client.peers

## Dataset Configuration

**DO4 LoRA Configuration:**
- 14 training samples
- 4 test samples
- LoRA rank 16 with alpha 32.0
- Enhanced training with augmentation

In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download

DATASET_DIR = Path("./dataset/").expanduser().absolute()

if not DATASET_DIR.exists():
    print("Downloading Chest CT Segmentation dataset...")
    snapshot_download(
        repo_id="khoaguin/chest-ct-segmentation",
        repo_type="dataset",
        local_dir=DATASET_DIR,
    )

DATASET_PATH = DATASET_DIR / "chest-ct-segmentation"
print(f"Dataset path: {DATASET_PATH}")

In [ ]:
# DO4 Configuration - LoRA Training
CLIENT_TYPE = "DO_4_lora"
TRAIN_SAMPLES = 14
TEST_SAMPLES = 4
LORA_RANK = 16
LORA_ALPHA = 32.0
LEARNING_RATE = 1e-4

print(f"Client Type: {CLIENT_TYPE}")
print(f"Training samples: {TRAIN_SAMPLES}")
print(f"Test samples: {TEST_SAMPLES}")
print(f"LoRA rank: {LORA_RANK}")
print(f"LoRA alpha: {LORA_ALPHA}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Augmentation: Enabled")
print(f"Contributes to aggregation: YES")

In [ ]:
# Upload dataset
do_client.create_dataset(
    name="chest-ct-segmentation",
    mock_path=DATASET_PATH / "mock" if (DATASET_PATH / "mock").exists() else DATASET_PATH,
    private_path=DATASET_PATH,
    summary=f"Chest CT Segmentation dataset for DO4 (LoRA: {TRAIN_SAMPLES} samples, rank {LORA_RANK})",
    readme_path=DATASET_PATH / "README.md" if (DATASET_PATH / "README.md").exists() else None,
    tags=["medical", "segmentation", "sam2", "ct", "lora"],
    sync=True,
)
print("Dataset uploaded!")

In [ ]:
do_client.datasets.get_all()

## Jobs

In [ ]:
do_client.jobs

In [ ]:
if len(do_client.jobs) > 0:
    print(do_client.jobs[0])

## Approve and Run Jobs

**Note**: DO4 performs full LoRA training and contributes weights to aggregation.

In [ ]:
if len(do_client.jobs) > 0:
    do_client.jobs[0].approve()
    print("Job approved!")

In [ ]:
do_client.process_approved_jobs()

In [ ]:
do_client.jobs

## Clean Up

In [ ]:
# do_client.delete_syftbox()